# 0.1 + 0.2 != 0.3

Written by:

* Philip Ilten (University of Cincinnati)

In this notebook we explore some common issues that arise when trying to implement math with computers.

## Requirements

Before getting started, we need to import a few modules. We will use `sys` and `platform` to determine system properties. We will use `fractions` and `decimal` for high-precision calculations. Finally, we will use `matplotlib` for plotting.

In [ ]:
# Import the `sys` and `platform` modules.
import sys, platform

# Import the `fraction` and `decimal` modules.
import fractions, decimal

# Import `matplotlib`.
import matplotlib.pyplot as plt

## Introduction

When doing calculations with computers, we oftentimes take for granted that the computer is doing what we think it should be doing. There's a reason for this, most of the time the computer *does* do what we think it should be doing. But, sometimes it doesn't and this behaviour can depend upon the setup used to produce the calculation: computer hardware (CPU, GPU, memory, *etc.*), operating system, compiler, programming language, and algorithm.

Let's start off with a simple example. First, we are going to break some things, so it would be good to know what our setup is.

In [ ]:
# What kind of CPU are we running on?
# The exclamation mark at the front in iPython runs a shell command
# outside of Python.
!lscpu
print("-" * 80)

# What about our operating system?
!cat /etc/os-release
print("-" * 80)

# What interpreter am I using?
print(platform.python_implementation())
print("-" * 80)

# What version of Python are we using?
!python --version
print("-" * 80)

This is a lot of information! In the end, we won't use most of it, but it is good to know that we are using `Python 3` on a 64-bit CPU. We will use this information a little bit later.

So, let's try an example. Will the following cell evaluate to `True` or `False`?

In [ ]:
1 + 2 == 3

Ok, so far, so good. Hopefully this is what we expect. Let's try this again!

In [ ]:
0.1 + 0.2 == 0.3

Well, that was unexpected. What's going on?

## Numerical Precision

The short answer is numerical precision. What about the slightly longer answer? First, what happens if we write out $1/3$ in decimal form?
$$
1/3 = 0.3\bar{3}
$$
We end up with our last digit $3$ repeating. We can't write $1/3$ in decimal form with a finite number of digits.

Now, let's do the same thing for $1/10$. But here's the catch, computers don't work in a base $10$ math system (decimal), but rather base $2$ (binary). So, we need to write $1/10$ in binary. How do we do that for a fractional decimal?

1. Take the fractional part of our decimal number, in this case just $0.1$ and multiply this by $2$.
$$
0.1\times2 = \color{red}{0}.2
$$
2. Take the digit to the left of the decimal point (highlighted in red) as our first binary digit.
$$
1/10 = 0.1 = 0.\color{red}{0}\ldots_2
$$
3. Subtract this digit from $0.2$. If $0$, we're done.
$$
0.2 - \color{red}{0} = 0.2
$$
   Looks like we're not done ...
4. If not $0$, return to step 1 but using this difference.
$$
\begin{aligned}
0.2\times2 &= \color{orange}{0}.4 \\
1/10 = 0.1 &= 0.0\color{orange}{0}\ldots_2 \\
0.4 - \color{orange}{0} &= 0.4 \\
0.4\times2 &= \color{yellow}{0}.8 \\
1/10 = 0.1 &= 0.00\color{yellow}{0}\ldots_2 \\
0.8 - \color{yellow}{0} &= 0.8 \\
0.8\times2 &= \color{green}{1}.6 \\
1/10 = 0.1 &= 0.000\color{green}{1}\ldots_2 \\
1.6 - \color{green}{1} &= 0.6 \\
0.6\times2 &= \color{blue}{1}.2 \\
1/10 = 0.1 &= 0.0001\color{blue}{1}\ldots_2 \\
1.2 - \color{blue}{1} &= 0.2 \\
0.2\times2 &= \color{purple}{0}.4 \\
1/10 = 0.1 &= 0.00011\color{purple}{0}\ldots_2 \\
\end{aligned}
$$

Wait, we're just back to $0.2$, this whole process will continue on forever! It turns out that just like $1/3$ in decimal, $1/10$ in binary cannot be expressed with a finite number of digits.
$$
1/10 = 0.0\overline{0011}_2
$$
If we want to store this number to full precision with a computer, we would need an infinite number of bits! How many do we have?

In [ ]:
# `getsizeof` of returns the number of bytes for an object.
total = sys.getsizeof(0.1)
# But, every CPython object has an overhead of 16 bytes.
overhead = 16
raw = total - overhead
# Finally, we need to go from bytes to bits.
# 8 bits = 1 byte
raw * 8

Well, that's certainly not infinite! Actually, we might have expected this given the info of the CPU we are running on, since it did tell us it was `64`-bit. Let's see the practical effect of this truncation.

In [ ]:
0.1 + 0.2

At the 17th digit, we run into a problem. Do we expect this, given what we know about representing these fractions in binary? Let us explore a little example where we see what happens when we perform fixed point addition. This is not quite what `Python` is doing with `0.1 + 0.2`, but very similar.

In [ ]:
###START_EXERCISE
# First, write a function that implements our binary conversion above.
def dec_to_bin(num, den, prec=10):
    """
    Convert the decimal fraction with numerator `num` and denominator `den`
    into a binary fraction with precision `prec` using the algorithm above. The
    result is a string to avoid numerical representation issues.
    """
    bin = ""
    # The `Fraction` class stores the numerator and denominator both as
    # integers.
    diff = fractions.Fraction(num, den)
    # Implement the algorithm described above here.
    # Step 1, multiply by 2.
    # Step 2, find the charateristic (integer to the left of the
    # decimal point), and append to the binary digits.
    # Step 3, take the difference.
    # Return the result (pad with zeros to the requested precision).
    return bin.ljust(prec, "0")


###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# First, we write a function that implements our binary conversion above.
def dec_to_bin(num, den, prec=10):
    """
    Convert the decimal fraction with numerator `num` and denominator `den`
    into a binary fraction with precision `prec` using the algorithm above. The
    result is a string to avoid numerical representation issues.
    """
    binary = ""
    # The `Fraction` class stores the numerator and denominator both as
    # integers.
    diff = fractions.Fraction(num, den)
    while len(binary) < prec and diff != 0:
        # Step 1, multiply by 2.
        diff *= 2
        # Step 2, find the charateristic (integer to the left of the
        # decimal point), and append to the binary digits.
        char = int(diff)
        binary += str(char)
        # Step 3, take the difference.
        diff -= char
    # Return the result (pad with zeros to the requested precision).
    return binary.ljust(prec, "0")


###STOP_SOLUTION

In [ ]:
###START_EXERCISE
# Second, write a function that can add our two binary strings together.
def add_bin(a, b):
    """
    Add two strings representing binary fractions together, `a + b`. This
    only returns the fractional component of the sum.
    """
    # Make sure both binary strings are the same length.
    # This is the sum of the two binary fraction strings.
    total = ""
    # This is what we carry over when adding each set of digits.
    quotient = 0
    # Loop over the bits backward.
    # Add the bits together.
    # Return the reversed string.
    return total[::-1]


###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# Second, we write a function that can add our two binary strings together.
def add_bin(a, b):
    """
    Add two strings representing binary fractions together, `a + b`. This
    only returns the fractional component of the sum.
    """
    # Make sure both binary strings are the same length.
    n = max(len(a), len(b))
    a, b = a.ljust(n, "0"), b.ljust(n, "0")
    # This is the sum of the two binary fraction strings.
    total = ""
    # This is what we carry over when adding each set of digits.
    quotient = 0
    # Loop over the bits backward.
    for aBit, bBit in zip(reversed(a), reversed(b)):
        # Add the bits together.
        quotient, remainder = divmod(quotient + int(aBit) + int(bBit), 2)
        total += str(remainder)
    # Return the reversed string.
    return total[::-1]


###STOP_SOLUTION

In [ ]:
###START_EXERCISE
# Third, write a function that converts our binary-fraction string back to a
# decimal-fraction string.
def bin_to_dec(bin):
    """
    Convert the binary-fraction string `bin` into a decimal-string fraction
    using the reverse of the `dec_to_bin` algorithm.
    """
    # We use the Decimal module for arbitrary precision.
    with decimal.localcontext() as ctx:
        # Set the precision sufficiently high.
        ctx.prec = len(bin) + 1
        # Start with 0.
        dec = decimal.Decimal(0)
        # Add each bit as 2^-i, where i is its index.
        return str(dec)


###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# Third, we write a function that converts our binary-fraction string back to a
# decimal-fraction string.
def bin_to_dec(bin):
    """
    Convert the binary-fraction string `bin` into a decimal-string fraction
    using the reverse of the `dec_to_bin` algorithm.
    """
    # We use the Decimal module for arbitrary precision.
    with decimal.localcontext() as ctx:
        # Set the precision sufficiently high.
        ctx.prec = len(bin) + 1
        # Start with 0.
        dec = decimal.Decimal(0)
        # Add each bit as 2^-i, where i is its index.
        for power, bit in enumerate(bin):
            dec += int(bit) / decimal.Decimal(2) ** (power + 1)
        return str(dec)


###STOP_SOLUTION

In [ ]:
###START_EXERCISE
# Now, we can use our functions above to see if we have a problem.
# Set the precision. Try 10 bits.
# Write out the two fractions.
# Add the fractions.
# Convert back to decimal.
###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# Set the precision. Here, we use 10 bits.
p = 10

# Write out the two fractions.
a = dec_to_bin(1, 10, p)
b = dec_to_bin(2, 10, p)
print("0.1:", a)
print("0.2:", b)

# Add the fractions.
c = add_bin(a, b)
print("sum:", c)

# Convert back to decimal.
d = bin_to_dec(c)
print("dec:", d)
###STOP_SOLUTION

Well, it looks like we certainly have a problem! Can we reproduce the same result as what we got above with `0.1 + 0.2 = 0.3`?

In [ ]:
###START_EXERCISE
# Use our functions to calculate the issue for 64 bits.
###STOP_EXERCISE

In [ ]:
###START_SOLUTION
p = 64
bin_to_dec(add_bin(dec_to_bin(1, 10, p), dec_to_bin(2, 10, p)))
###STOP_SOLUTION

Interesting, if we assume `64` bits, then we see a difference at the 20th digit. So, `Python` must be using less than `64` bits to represent this number. How many then?

In [ ]:
###START_EXERCISE
# Loop over precisions.
# Get the decimal string.
# Find the first non-nine entry.
###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# Loop over precisions.
for p in range(10, 65):
    # Get the decimal string.
    d = bin_to_dec(add_bin(dec_to_bin(1, 10, p), dec_to_bin(2, 10, p)))
    # Find the first non-nine entry.
    for i in range(3, len(d)):
        if d[i] != "9":
            break
    print(p, i - 1, d)
###STOP_SOLUTION

It looks like the answer is `53`. It turns out that `Python` implements the [754-2019 - IEEE Standard for Floating-Point Arithmetic](https://ieeexplore.ieee.org/document/8766229) which corresponds to a `53`-bit fractional component of the number.

## Floating Points

What happened to the other `11` bits? Here, we have only considered fractional numbers, where the radix point (`.`) is always at the start of the number. The IEEE standard is a "floating-point" representation, where the number is effectively stored in scientific notation but in binary not decimal.

$$
\begin{array}{rllll}
6.5
=& &  & \underbrace{6.5}_{\text{significand}} & \times & \overbrace{10}^{\text{base}}\underbrace{^{0}}_{\text{exponent}} \\
=& \underbrace{(-1)^0}_{\text{sign}} & \times & \underbrace{1.101_2}_{\text{significand}} & \times & \overbrace{2}^{\text{base}}\underbrace{^{2}}_{\text{exponent}}
\end{array}
$$

So, these additional `11` bits store the sign and the exponent of the number, while the `53` bits store the significand. But, since the first number of the significand is always `1`, we can cheat and use this bit for the exponent!

* `1` bit for sign
* `52` bits for significand
* `1` bit for exponent sign
* `10` bits for exponent

Knowing this format, what is approximately the largest number we can represent?

In [ ]:
# We have a maximum exponent of 2^10, so we can calculate what
# our maximum exponent is in decimal.
len(str(2 ** (2**10))) - 1

But wait, how can we do the calculation above, if this goes beyond the limit of our representation?

In [ ]:
# When we write 2, Python interprets this as an integer.
print("2:", type(2))

# When we write 2.0, Python interprets this as a float.
print("2.0:", type(2.0))

In [ ]:
###START_EXERCISE
# Redo the calculation above using floats.
###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# We can redo the calculation using a float and see what happens.
try:
    len(str(2.0 ** (2.0**10))) - 1
except Exception as e:
    print(e)
###STOP_SOLUTION

This means that `Python` doesn't have the same limitation for integers somehow. Why not? This is because we can always represent a decimal integer with a finite number of bits in binary. So in `Python`, integers are not represented by a fixed number of bits but rather the number of bits are expanded as necessary. This is called "big number" or "bignum" representation.

In [ ]:
###START_EXERCISE
# Loop over some different integer values and find how many bits they use.
###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# Let's see this in action! Every CPython object has an overhead of 16 bytes.
overhead = 16
# Loop over different integers.
for i in range(0, 500, 100):
    print(f"2**{i} [bits]:", (sys.getsizeof(2**i) - overhead) * 8)
###STOP_SOLUTION

## Patriot Missiles

Now we can explore a real world example of a numerical precision problem. The MIM-104 was a US Army surface-to-air, mobile, air defense missile system used during the first Gulf War.
![MIM-104 launch](https://upload.wikimedia.org/wikipedia/commons/f/f8/Patriot_missile_launch_b.jpg)

In 1991, an MIM-104 failed to intercept a Scud missile which hit a US Army base. The cause was a numerical precision issue, detailed in [GAO/IMTEC-92-96](https://www.gao.gov/assets/imtec-92-26.pdf).

What went wrong? An integer time, stored as tenths of seconds, was converted into a fractional 23-bit value (the register itself was 24 bits). We can calculate the error this introduces for every tenth of a second.

In [ ]:
###START_EXERCISE
# Use bin_to_dec and dec_to_bin to calculate this error.
###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# This calculates the error per tenth of a second.
error_tenth = 1 / 10 - float(bin_to_dec(dec_to_bin(1, 10, 23)))
print("error on tenth of a second [s]:", error_tenth)
###STOP_SOLUTION

This may not seem like a lot, but the MIM-104 that failed to intercept the Scud missile was operational for 100 hours. With our previous result, we can find the error in the time the system had accumulated.

In [ ]:
###START_EXERCISE
# Calculate the error for 100 hours of operation.
###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# 100 hours * 60 minutes * 60 seconds * 10 tenths.
error_time = 100 * 60 * 60 * 10 * error_tenth
print("error for 100 hour operation [s]:", error_time)
###STOP_SOLUTION

After initially detecting a missile the MIM-104 tracks the missile in a "range gate area". If the missile falls ouside of this area, the system cannot track it. Scud missile have velocities of around $2000$ meters/second, and so we can calculate how far off this range gate was.

In [ ]:
###START_EXERCISE
# Calculate the range gate error.
###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# We multiple our time error by the missile velocity.
error_gate = error_time * 2000
print("range gate error [m]:", error_gate)
###STOP_SOLUTION

The range gate was miscalculated by over half a kilometer.

## Big Data

Sometimes not having enough precision causes problems, but sometimes the opposite is also true. Ideally, we represent our numbers with just the right amount of precision. In particle physics, this can be particularly important.

The CODEX-$\beta$ demonstrator on the Large Hadron Collider (LHC) was a small experiment that used Resistive Plate Chambers (RPCs), devices that detect electric singals from gas ionized by charged particles. Each RPC has $96$ channels and $3$ RPCs are stacked together to produce a module with $256$ channels.

Every $25$ nanoseconds each module is read out (a rate of $40$ MHz). A boolean is `True` or `False` and is just one bit. Every time we read out the module, we could store a boolean for each channel, corresponding to whether it fired or not.

In [ ]:
###START_EXERCISE
# Calculate this rate in gigabytes/second (GB/s).
###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# We can calculate the rate in gigabytes/second (GB/s).
# (read out size)/(read out time)/(convert to GB)
(256) / (25 * 10**-9) / (8 * 10**9)
###STOP_SOLUTION

Well, if we take data for any extended period, that's a lot of data. What if we only store the channel ID (`1` - `256`) for channels that have fired, and assume we have two channels fire per $25$ ns read out. Let's also store the channel ID as a float.

In [ ]:
###START_EXERCISE
# Calculate this updated rate in gigabytes/second (GB/s).
###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# We can calculate this updated rate in gigabytes/second (GB/s).
# (read out size)/(read out time)/(convert to GB)
(2 * 64) / (25 * 10**-9) / (8 * 10**9)
###STOP_SOLUTION

Ok, so that is certainly better, but we are storing way more precision than we need. How much precision do we need?
$$
2^n = 256 \to n = \log(256)/\log(2) = 8
$$

In [ ]:
###START_EXERCISE
# Calculate this updated rate in gigabytes/second (GB/s).
###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# We can calculate this updated rate in gigabytes/second (GB/s).
# (read out size)/(read out time)/(convert to GB)
(2 * 8) / (25 * 10**-9) / (8 * 10**9)
###STOP_SOLUTION

This is much better! Then of course we can apply compression, *etc.*, but in the end, the representation that we choose for our data is a balance between too much precision and too little precision.

What happens when we write our data to a text file? Well, text files typically use ASCII encoding where `128` characters are defined. This requires `7` bits to store, per character.
$$
2^n = 128 \to n = \log(128)/\log(2) = 7
$$

It turns out `7` bits is rather awkward for most CPUs, which prefer minimum groupings of `1` byte (`8` bits), and so `8` bits is used instead.
This means that whenever we store numerical data with ASCII, we take a penalty in storage of $\times8$. So, bottom line, if you need to store lots of numerical data, don't use text files!

Another very clear example are Large Language Models (LLM) which typically store and manipulate model weights using `32`-bit floating-point representation. In 2024 the DeepSeek-V3 shocked the community as it was as performant as other leading models of the time from OpenAI and Anthropic, but required significantly less GPU hours to train. One of the reasons is attrubuted to using an `8`-bit representation, see [arXiv:2412.19437](https://arxiv.org/abs/2412.19437).

## Lorentz Boosts

We have seen how representation is important, but can we always be guaranteed that a given representation is sufficient? In particle physics we use special relavity to transform from one reference frame to another reference frame. We call these "Lorentz boosts".

Usually, we are more interested about the momentum and energy of the particle, rather than the position and time of the particle. We use these momentum boosts all the time in particle physics analyses.

$$
\begin{aligned}
E' &= \gamma(E - \beta pc) \\
p' &= \gamma(p - \beta E/c) \\
\end{aligned}
$$

Here, $E$ is the particle energy and $p$ is the particle momentum. The unprimed frame is our original frame, and the primed frame is our boosted frame. The Lorentz factor $\gamma$ and normalized velocity $\beta$ are parameters set by our boost.

We oftentimes want to boost our initial particle into the rest frame of another particle $b$, where $b$ has no momentum. We can write out $\gamma$ and $\beta$ for this boost.
$$
\begin{aligned}
\beta &= -p_b/E_b \\
\gamma &= 1/\sqrt{1 - \beta^2}
\end{aligned}
$$

But as we boost into the frame of particle $b$ with more and more momentum, $\beta \to 1$. This means for $\gamma$ we are effectively calculating $1 - 1$ in the denominator. Here, a "catastrophic cancellation" occurs because we are subtracting two numbers that are almost the same, with a result smaller than our numerical precision.

Luckily, we can avoid this, by reworking $\gamma$ with a little bit of algebra and the energy-momentum relation, $E^2 = p^2 + m^2$.
$$
\gamma = E_b/m_b
$$
We can now write a function that performs this boost.

In [ ]:
###START_EXERCISE
# Write a function that implements this boost.
def boost(E, p, p_b, m_b, stable=False, dec=None):
    """
    Boost a particle with energy `E` and momentum `p` into the frame of
    particle `b` with momentum `p_b` and mass `m_b`. Returns E' and p'.
    If `stable` is `True`, calculate `gamma` as `E_b/m_b`. If `dec`
    is an integer, use the `Decimal` package with this precision.
    """
    # We use the Decimal module for arbitrary precision.
    with decimal.localcontext() as ctx:
        # Use decimal representation and set the precision.
        if type(dec) is int:
            rep = decimal.Decimal
            ctx.prec = dec
        # Use floating-point representation otherwise.
        else:
            rep = float
        # Convert all our values to Decimal or float.
        E, p, p_b, m_b = rep(E), rep(p), rep(p_b), rep(m_b)
        # Calculate the boost energy.
        # Calculate the beta factor.
        # Calculate the gamma factor (either stable or unstable).
        # Perform the boost.
        # Return the result.


###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# Here, we write a function that implements this boost.
def boost(E, p, p_b, m_b, stable=False, dec=None):
    """
    Boost a particle with energy `E` and momentum `p` into the frame of
    particle `b` with momentum `p_b` and mass `m_b`. Returns E' and p'.
    If `stable` is `True`, calculate `gamma` as `E_b/m_b`. If `dec`
    is an integer, use the `Decimal` package with this precision.
    """
    # We use the Decimal module for arbitrary precision.
    with decimal.localcontext() as ctx:
        # Use decimal representation and set the precision.
        if type(dec) is int:
            rep = decimal.Decimal
            ctx.prec = dec
        # Use floating-point representation otherwise.
        else:
            rep = float
        # Convert all our values to Decimal or float.
        E, p, p_b, m_b = rep(E), rep(p), rep(p_b), rep(m_b)
        # Calculate the boost energy.
        E_b = (p_b**2 + m_b**2) ** rep("0.5")
        # Calculate the beta factor.
        beta = -p_b / E_b
        # Calculate the gamma factor.
        if stable:
            # This does not have a catastrophic cancellation.
            gamma = E_b / m_b
        else:
            # This does, to the point where we need to catch for divide by 0.
            try:
                gamma = 1 / (1 - beta**2) ** rep("0.5")
            except:
                gamma = rep("inf")
        # Perform the boost.
        E_p = gamma * (E - beta * p)
        p_p = gamma * (p - beta * E)
        # Return the result.
        return float(E_p), float(p_p)


###STOP_SOLUTION

We can now use this function to check how big an issue this is.

In [ ]:
###START_EXERCISE
# Mass of the particle whose frame we are boosting (electron mass in GeV).
m_b = 0.000511
# Momentum of the particle whose frame we are boosting.
p_bs = [p_b for p_b in range(1000)]
# Energy and momentum of the particle we are boosting.
E, p = m_b, 0
# Plot the difference between numerically unstable and high precision.
# Plot the difference between stable and high precision.
# Label the plot.
###STOP_EXERCISE

In [ ]:
###START_SOLUTION
# Mass of the particle whose frame we are boosting (electron mass in GeV).
m_b = 0.000511
# Momentum of the particle whose frame we are boosting.
p_bs = [p_b for p_b in range(1000)]
# Energy and momentum of the particle we are boosting.
E, p = m_b, 0
# Plot the difference between numerically unstable and high precision.
plt.plot(
    p_bs,
    [
        abs(boost(E, p, p_b, m_b, False)[1] - boost(E, p, p_b, m_b, True, 100)[1])
        for p_b in p_bs
    ],
    label="unstable",
)
# Plot the difference between stable and high precision.
plt.plot(
    p_bs,
    [
        abs(boost(E, p, -p_b, m_b, True)[1] - boost(E, p, -p_b, m_b, True, 100)[1])
        for p_b in p_bs
    ],
    "--",
    label="stable",
)
# Label the plot.
plt.xlabel(r"$p_B$ [GeV]")
plt.ylabel(r"$p' - p'_\text{true}$ [GeV]")
plt.legend()
###STOP_SOLUTION

Certainly at large boosts we start to run into issues! This occurs as $p_b \to \infty$ or $m_b \to 0$. Of course this numerical stability trick only works if we know the mass of the particle whose frame we are boosting into.